In [1]:
import os
import pandas as pd
from skimage.feature import hog
from skimage.color import rgb2gray, rgb2hsv
from skimage.transform import resize
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from PIL import Image
import numpy as np
import tqdm
import tensorflow as tf
from tensorflow.keras import layers, models

2026-05-14 11:38:10.215678: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-14 11:38:10.343843: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# Create dataframes from dataset files


DATA_DIR = "CUB_200_2011_Subset20classes"
#DATA_DIR = "CUB_200_2011"
IMAGES_DIR = os.path.join(DATA_DIR, "images")

# Load the metadata files
images_df = pd.read_csv(
    os.path.join(DATA_DIR, "images.txt"),
    sep=" ",
    names=["image_id", "image_path"]
)

labels_df = pd.read_csv(
    os.path.join(DATA_DIR, "image_class_labels.txt"),
    sep=" ",
    names=["image_id", "class_id"]
)

bbox_df = pd.read_csv(
    os.path.join(DATA_DIR, "bounding_boxes.txt"),
    sep=" ",
    names=["image_id", "x", "y", "width", "height"]
)

splt_df = pd.read_csv(
    os.path.join(DATA_DIR, "train_test_split.txt"),
    sep=" ",
    names=["image_id", "is_train"]
)

# Merge all the dataframes into a single frame called 'data'
data = images_df.merge(labels_df, on="image_id").merge(bbox_df, on="image_id").merge(splt_df, on="image_id")

unique_classes = sorted(data["class_id"].unique())
print(f"Number of unique classes: {len(unique_classes)}")

class_mapping = {old: new for new, old in enumerate(unique_classes)}

data["label"] = data["class_id"].map(class_mapping)

data["full_path"] = data["image_path"].apply(
    lambda x: os.path.join(IMAGES_DIR, x)
)

# Split data into training and testing sets
train_df = data[data["is_train"] == 1].reset_index(drop=True)
test_df = data[data["is_train"] == 0].reset_index(drop=True)

train_paths = train_df["full_path"].values
train_labels = train_df["label"].values
train_bboxes = train_df[["x", "y", "width", "height"]].values

test_paths = test_df["full_path"].values
test_labels = test_df["label"].values
test_bboxes = test_df[["x", "y", "width", "height"]].values

print("Training samples:", len(train_df))
print("Testing samples:", len(test_df))
print("Number of classes:", data["label"].nunique())

Number of unique classes: 20
Training samples: 892
Testing samples: 223
Number of classes: 20


In [3]:
# Calculate most common size size of images in the dataset
most_common_width = int(data["image_path"].apply(lambda x: Image.open(os.path.join(IMAGES_DIR, x)).size[0]).mode()[0])
most_common_height = int(data["image_path"].apply(lambda x: Image.open(os.path.join(IMAGES_DIR, x)).size[1]).mode()[0])
most_common_size = (most_common_width, most_common_height)
print(f"Most common image size: {most_common_size}")

# While the most common size is (500, 500), I run out of RAM, so we'll need to need to resize the images to a smaller size like (300, 300)
most_common_size = (250, 250)


# Resize the image to a fixed size for consistent feature extraction
def resizeImage(image, target_size):
    return resize(image, target_size)


# Extract 
def extract_hog(image):
    
    # Convert to grayscale (HOG works on intensity gradients)
    gray = rgb2gray(image)

    # Extract HOG features (edge/shape descriptors)
    features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2)
    )
    return features

def extract_HSV_histogram(image):
    # Convert to HSV color space
    hsv = rgb2hsv(image)

    # Compute histograms for each channel
    h_hist, _ = np.histogram(hsv[:, :, 0], bins=50, range=(0, 1))
    s_hist, _ = np.histogram(hsv[:, :, 1], bins=50, range=(0, 1))
    v_hist, _ = np.histogram(hsv[:, :, 2], bins=50, range=(0, 1))

    # Concatenate histograms into a single feature vector
    histogram_features = np.concatenate([h_hist, s_hist, v_hist])
    return histogram_features

# This function builds the dataset (X, y)
def build_dataset(paths, labels, avg_size):
    X, y = [], []
    for p, l in tqdm.tqdm(zip(paths, labels), total=len(paths), desc="Building SVM dataset"):
        # Extract feature for each image
        image = Image.open(p).convert("RGB")
        image = np.array(image)
        image_data = resizeImage(image, avg_size)
        X.append(np.concatenate((
            extract_hog(image_data), 
            extract_HSV_histogram(image_data)
            )))
        y.append(l)
    return np.array(X), np.array(y)

# Build training and testing feature sets
X_train, y_train = build_dataset(train_paths, train_labels, avg_size=most_common_size)
X_test, y_test = build_dataset(test_paths, test_labels, avg_size=most_common_size)




Most common image size: (500, 500)


Building SVM dataset: 100%|██████████| 223/223 [00:15<00:00, 14.22it/s]


In [4]:
# Train a SVM on the features
svm = SVC(
    kernel="linear", C=1)
svm.fit(X_train, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [5]:
# Evaluate SVM on test set
svm_accuracy = svm.score(X_test, y_test)
print(f"SVM Overall Accuracy: {svm_accuracy:.4f}")

# SVM class based accuracy
svm_predictions = svm.predict(X_test)
class_accuracies = {}
for cls in unique_classes:
    cls_indices = np.where(y_test == cls)[0]
    if len(cls_indices) == 0:
        print(f"SVM Accuracy for Class {cls}: no test samples")
        class_accuracies[cls] = np.nan
        continue
    cls_accuracy = np.mean(svm_predictions[cls_indices] == y_test[cls_indices])
    class_accuracies[cls] = cls_accuracy
    print(f"SVM Accuracy for Class {cls}: {cls_accuracy:.4f}")

# Optional: show how many test samples contributed to each class accuracy
print("\nTest samples per class:")
for cls in unique_classes:
    print(f"Class {cls}: {np.sum(y_test == cls)}")


SVM Overall Accuracy: 0.1614
SVM Accuracy for Class 1: 0.1667
SVM Accuracy for Class 2: 0.0833
SVM Accuracy for Class 3: 0.2500
SVM Accuracy for Class 4: 0.0000
SVM Accuracy for Class 5: 0.0000
SVM Accuracy for Class 6: 0.1000
SVM Accuracy for Class 7: 0.4000
SVM Accuracy for Class 8: 0.0000
SVM Accuracy for Class 9: 0.0833
SVM Accuracy for Class 10: 0.0833
SVM Accuracy for Class 11: 0.0909
SVM Accuracy for Class 12: 0.1667
SVM Accuracy for Class 13: 0.0833
SVM Accuracy for Class 14: 0.1667
SVM Accuracy for Class 15: 0.0909
SVM Accuracy for Class 16: 0.3636
SVM Accuracy for Class 17: 0.2222
SVM Accuracy for Class 18: 0.0000
SVM Accuracy for Class 19: 0.0000
SVM Accuracy for Class 20: no test samples

Test samples per class:
Class 1: 12
Class 2: 12
Class 3: 12
Class 4: 9
Class 5: 8
Class 6: 10
Class 7: 10
Class 8: 12
Class 9: 12
Class 10: 12
Class 11: 11
Class 12: 12
Class 13: 12
Class 14: 12
Class 15: 11
Class 16: 11
Class 17: 9
Class 18: 12
Class 19: 12
Class 20: 0


In [ ]:
# Create dataset for CNN
def build_cnn_dataset(paths, labels, avg_size):
    X, y = [], []
    for p, l in tqdm.tqdm(zip(paths, labels), total=len(paths), desc="Building CNN dataset"):
        image = Image.open(p).convert("RGB")
        image = np.array(image)
        image_data = resizeImage(image, avg_size)
        X.append(image_data)
        y.append(l)
    return np.array(X), np.array(y)
X_train, y_train = build_cnn_dataset(train_paths, train_labels, avg_size=most_common_size)
X_test, y_test = build_cnn_dataset(test_paths, test_labels, avg_size=most_common_size)

In [8]:

# CNN
def cnn(NUM_CLASSES):
    # Define simple CNN architecture
    model = models.Sequential([
        layers.Input((250,250,3)),

        # Learn low-level features (edges, textures)
        layers.Conv2D(32, 5, activation='relu'),
        layers.MaxPooling2D(),

        # Learn more complex patterns
        layers.Conv2D(64, 3, activation='relu'),
        layers.MaxPooling2D(),

        # Learn higher-level object structures
        layers.Conv2D(128, 3, activation='relu'),
        layers.MaxPooling2D(),

        # Learn even more abstract features
        layers.Conv2D(256, 3, activation='relu'),
        layers.MaxPooling2D(),

        # Pool spatial features before the dense layer
        layers.GlobalAveragePooling2D(),

        # Fully connected layer for classification
        layers.Dense(64, activation='relu'),

        # Output layer (20 bird classes)
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])

    # Compile model
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Reserve the existing test set for final evaluation and create a validation split from training data
X_train_cnn, X_val_cnn, y_train_cnn, y_val_cnn = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

NUM_CLASSES = len(unique_classes)
model = cnn(NUM_CLASSES)
model.fit(
    X_train_cnn,
    y_train_cnn,
    epochs=30,
    batch_size=32,
    validation_data=(X_val_cnn, y_val_cnn)
)


Epoch 1/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 23s 958ms/step - accuracy: 0.0463 - loss: 2.9984 - val_accuracy: 0.0503 - val_loss: 2.9943
Epoch 2/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 20s 872ms/step - accuracy: 0.0519 - loss: 2.9951 - val_accuracy: 0.0782 - val_loss: 2.9933
Epoch 3/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 21s 919ms/step - accuracy: 0.0659 - loss: 2.9901 - val_accuracy: 0.0838 - val_loss: 2.9761
Epoch 4/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 19s 826ms/step - accuracy: 0.0659 - loss: 2.9743 - val_accuracy: 0.0726 - val_loss: 2.9699
Epoch 5/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 20s 870ms/step - accuracy: 0.0771 - loss: 2.9331 - val_accuracy: 0.0726 - val_loss: 2.9004
Epoch 6/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 20s 888ms/step - accuracy: 0.0982 - loss: 2.8858 - val_accuracy: 0.0782 - val_loss: 2.8740
Epoch 7/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 19s 828ms/step - accuracy: 0.1136 - loss: 2.8486 - val_accuracy: 0.0894 - val_loss: 2.8621
Epoch 8/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 19s 836ms/step - accuracy: 0.1136 - loss: 2.8090 - val_accu

In [ ]:
# So i don't need to store the massive dataframs for later analysis
# the models are saved and the reopened in the next cell for evaluation and analysis

# Save the models
model.save("cnn_model.keras")
import joblib
joblib.dump(svm, "svm_model.joblib")

['svm_model.joblib']

In [ ]:
# Open the models
from tensorflow.keras.models import load_model
cnn_model = load_model("cnn_model.keras")
svm_model = joblib.load("svm_model.joblib")

# Analyze the models (accuracy, confusion matrix, class-wise performance, etc.)
from sklearn.metrics import classification_report, confusion_matrix
cnn_predictions = np.argmax(cnn_model.predict(X_test), axis=1)
print("CNN Classification Report:")
print(classification_report(y_test, cnn_predictions))
print("CNN Confusion Matrix:")
print(confusion_matrix(y_test, cnn_predictions))